In [ ]:
# =====================================================================
# CELL 2: IMPORT LIBRARY & LOAD MODEL
# =====================================================================
import os
import time
import joblib
import numpy as np
import scipy.io.wavfile as wav
import sounddevice as sd
import torch
import re
import pandas as pd
from faster_whisper import WhisperModel

# 1. LOAD PIPELINE MODEL CLASSIFICATION ARYA (6 TARGET)
MODEL_PATH = "../../../models/ticketing/model_tfidf_rf_exigen.pkl"

print("⏳ Memuat model klasifikasi tiket pintar Exigen...")
if os.path.exists(MODEL_PATH):
    pipeline_nlp = joblib.load(MODEL_PATH)
    print("✅ Model NLP berhasil dimuat dari folder lokal!")
else:
    raise FileNotFoundError(
        f"⚠️ Model tidak ditemukan di jalur: {MODEL_PATH}. Pastikan file .pkl sudah diletakkan dengan benar."
    )

# 2. INISIALISASI MODEL WHISPER INT8 & VAD FILTER
print("\n⏳ Memuat Model Whisper (Varian: Base, Tipe Komputasi: int8)...")
DEVICE_HARDWARE = "cuda" if torch.cuda.is_available() else "cpu"
model_stt = WhisperModel("base", device=DEVICE_HARDWARE, compute_type="int8")
print(f"✅ Model Whisper siap tempur di perangkat: {DEVICE_HARDWARE}!")

In [ ]:
# =====================================================================
# CELL 3: PREPROCESSING & KAMUS SLANG
# =====================================================================
print("⏳ Mengunduh Kamus Bahasa Gaul (Slang Dictionary)...")
url_slang = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
try:
    df_slang = pd.read_csv(url_slang)
    slang_dict_external = dict(zip(df_slang['slang'], df_slang['formal']))
    print(f"✅ Berhasil memuat {len(slang_dict_external)} kata slang/gaul!")
except Exception as e:
    print(f"⚠️ Gagal mengunduh kamus slang. Error: {e}")
    slang_dict_external = {} 

def super_clean_text(text):
    text = text.lower()
    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text) # Hapus typo stuttering
    text = re.sub(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\s*([a-z0-9]+)\b', r'gedung_\2', text)
    text = re.sub(r'\b(lantai|lt\.?|level)\s*([a-z0-9]+)\b', r'lantai_\2', text)
    text = re.sub(r'\b(ruang|rg\.?|kamar|kmr)\s*([a-z0-9]+)\b', r'ruang_\2', text)
    text = re.sub(r'[^a-z0-9_]', ' ', text).strip()
    
    kata_kata = text.split()
    kata_normal = [slang_dict_external.get(kata, kata) for kata in kata_kata]
    return " ".join(kata_normal)

In [ ]:
# =====================================================================
# CELL 4: FUNGSI AUDIO STREAMING & TRANSKRIPSI
# =====================================================================

def rekam_suara_mikrofon(nama_file="laporan_suara_temp.wav", durasi_maksimal=7, samplerate=16000):
    print(f"\n🎙️ [AUDIO] Mikrofon AKTIF... Silakan bicara (Maksimal {durasi_maksimal} detik).")
    print("🔴 PEREKAMAN DIMULAI...")
    rekaman = sd.rec(int(durasi_maksimal * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait() 
    print("🟢 Perekaman selesai.")
    wav.write(nama_file, samplerate, rekaman)
    return nama_file

def pipeline_speech_to_text(audio_path):
    if not os.path.exists(audio_path): return ""
    start_time = time.time()
    
    segments, info = model_stt.transcribe(
        audio_path, language="id", beam_size=5, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500)
    )
    
    hasil_teks = [segment.text for segment in segments]
    duration = time.time() - start_time
    
    print(f"⏱️ Deteksi Bahasa: {info.language} (Akurasi: {info.language_probability:.2%})")
    print(f"⏱️ Waktu STT     : {duration:.2f} detik")
    return " ".join(hasil_teks).strip()

In [ ]:
# =====================================================================
# CELL 5: EKSEKUSI PIPELINE (VOICE -> TEXT -> PREDICT TARGET)
# =====================================================================

# 1. Mulai Merekam Suara Anda
file_audio = rekam_suara_mikrofon(durasi_maksimal=7)

# 2. Transkripsi Audio ke Teks Mentah
print("\n🔀 Memulai proses transkripsi teks...")
teks_mentah = pipeline_speech_to_text(file_audio)
print(f"💬 Teks Mentah : \"{teks_mentah}\"")

# 3. Prediksi Rute Tiket Fase 1
if teks_mentah:
    teks_bersih = super_clean_text(teks_mentah)
    print(f"✨ Teks Bersih : \"{teks_bersih}\"")
    
    print("\n🤖 Meminta Model Multi-Output memprediksi...")
    tebakan_ai = pipeline_nlp.predict([teks_bersih])
    
    print("\n" + "=" * 50)
    print("🎯 TEBAKAN OTOMATIS TIKET PINTAR FASE 1 (STATUS: OPEN)")
    print("=" * 50)
    print(f" 🔹 Tipe Aset          : {tebakan_ai[0][0]}")
    print(f" 🔹 Lokasi Gedung      : {tebakan_ai[0][1]}")
    print(f" 🔹 Lokasi Lantai      : {tebakan_ai[0][2]}")
    print(f" 🔹 Lokasi Zona        : {tebakan_ai[0][3]}")
    print(f" 🔹 Kategori Dept      : {tebakan_ai[0][4]}")
    print(f" 🔹 Severity (Awal)    : {tebakan_ai[0][5]}")
    print("=" * 50)
    
    if tebakan_ai[0][5] in ["Berat", "Fatal", "Tinggi"]:
        print("🚨 ALERT: Severity TINGGI! Memicu WhatsApp API ke teknisi.")
else:
    print("❌ Suara tidak terdeteksi jelas.")